In [ ]:
from google.colab import files
uploaded = files.upload()

Saving mobilenetv2_plant.pth to mobilenetv2_plant.pth


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving leaf.JPG to leaf.JPG


In [ ]:
import torch
from torchvision import models
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.mobilenet_v2(weights=None)

model.classifier[1] = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, 38)
)

model.load_state_dict(torch.load("mobilenetv2_plant.pth", map_location=device))

model.to(device)
model.eval()

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [ ]:
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
image = Image.open("leaf.JPG").convert("RGB")

input_tensor = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    output = model(input_tensor)
    probs = torch.softmax(output, dim=1)[0]
    pred_idx = torch.argmax(probs).item()

print("Predicted class index:", pred_idx)
print("Confidence:", float(probs[pred_idx]))

Predicted class index: 37
Confidence: 0.9980733394622803


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving disease_protocols.json to disease_protocols.json


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving intelligence_engine.py to intelligence_engine.py


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — FULL INTELLIGENCE PIPELINE + VALIDATION ENGINE               ║
# ║  Self-contained — NO need to upload intelligence_engine.py              ║
# ║  Just paste this entire cell into Colab and run it                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# FILES YOU STILL NEED TO UPLOAD TO COLAB:
#   ✅ mobilenetv2_plant.pth
#   ✅ leaf.jpg  (or any leaf image)
#   ✅ disease_protocols.json
#   ❌ intelligence_engine.py  ← NOT needed anymore, all logic is here
# ─────────────────────────────────────────────────────────────────────────

import json, os, datetime, io, requests
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from PIL import Image

# ══════════════════════════════════════════════════════════════════════════
# ⚙️ USER CONFIGURATION — Edit these values
# ══════════════════════════════════════════════════════════════════════════

PROTOCOLS_PATH     = "disease_protocols.json"
CROP_AREA_ACRES    = 2.0      # How many acres your farm is
MARKET_PRICE_RS    = 1500.0   # Current mandi price (₹ per quintal)
TOP_K              = 3        # How many top predictions to show

# Location — get from Priyanshu's GPS module. Set manually for now:
LATITUDE           = None     # e.g. 21.1458  (or None to skip)
LONGITUDE          = None     # e.g. 79.0882  (or None to skip)

# Supabase DB — leave blank to skip DB insert (dry-run mode)
SUPABASE_URL       = ""       # "https://your-project.supabase.co"
SUPABASE_KEY       = ""       # "your-service-key"

# OpenWeatherMap — leave blank for static weather advice
OPENWEATHER_KEY    = ""       # "your-api-key"

# ══════════════════════════════════════════════════════════════════════════
# CLASS NAMES (38 PlantVillage classes — must match training order)
# ══════════════════════════════════════════════════════════════════════════

CLASS_NAMES = [
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust",
    "Apple___healthy", "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew", "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot", "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight", "Corn_(maize)___healthy",
    "Grape___Black_rot", "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot",
    "Peach___healthy", "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy",
    "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
    "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch", "Strawberry___healthy", "Tomato___Bacterial_spot",
    "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus", "Tomato___healthy",
]

# ══════════════════════════════════════════════════════════════════════════
# DISEASE TYPE DETECTOR  (dynamic — no hard-coded disease names)
# ══════════════════════════════════════════════════════════════════════════

_FUNGAL    = {"blight","mold","scab","mildew","rot","spot","rust","esca","measles","scorch"}
_BACTERIAL = {"bacterial"}
_VIRAL     = {"virus","viral","curl","mosaic","greening","haunglongbing"}
_PEST      = {"mite","spider"}
_BIO_WEIGHT = {"viral":1.25, "bacterial":1.10, "fungal":1.00, "pest":0.90,
               "healthy":0.0, "unknown":0.85}

def detect_context(disease_key):
    k = disease_key.lower()
    if "healthy" in k:
        return {"is_healthy": True,  "disease_type": "healthy",   "bio_weight": 0.0}
    if any(x in k for x in _VIRAL):
        return {"is_healthy": False, "disease_type": "viral",     "bio_weight": 1.25}
    if any(x in k for x in _BACTERIAL):
        return {"is_healthy": False, "disease_type": "bacterial", "bio_weight": 1.10}
    if any(x in k for x in _PEST):
        return {"is_healthy": False, "disease_type": "pest",      "bio_weight": 0.90}
    if any(x in k for x in _FUNGAL):
        return {"is_healthy": False, "disease_type": "fungal",    "bio_weight": 1.00}
    return {"is_healthy": False, "disease_type": "unknown", "bio_weight": 0.85}

# ══════════════════════════════════════════════════════════════════════════
# VALIDATION ENGINE — 8 strict consistency rules
# ══════════════════════════════════════════════════════════════════════════

def derive_severity(is_healthy, bio_weight, confidence):
    """Rule 4: Dynamic severity. Never static. Based on disease type × confidence."""
    if is_healthy:
        return "✅ NONE", "NONE"
    eff = confidence * bio_weight
    if eff >= 0.85:
        return "🔴 HIGH — Significant risk. Immediate intervention required.", "HIGH"
    elif eff >= 0.55:
        return "🟡 MEDIUM — Moderate risk. Treat within 48 hours and monitor daily.", "MEDIUM"
    else:
        return "🟢 LOW — Early-stage detection. Preventive treatment advised.", "LOW"

def validate_result(result, context):
    """
    Applies all 8 consistency rules to the raw result dict.
    Modifies and returns corrected result.
    """
    is_healthy   = context["is_healthy"]
    bio_weight   = context["bio_weight"]
    confidence   = context["confidence_raw"]

    # ── Rule 4: Derive severity dynamically ──────────────────────
    sev_str, sev_level = derive_severity(is_healthy, bio_weight, confidence)
    result["severity"]       = sev_str
    result["severity_level"] = sev_level

    # ── Rule 2 + 5: Healthy → force preventive-only ───────────────
    if is_healthy:
        result["first_aid"] = (
            "✅ No treatment required. Your plant is healthy. "
            "Continue regular monitoring and maintain good agronomic practices."
        )
        # Keep only preventive action steps
        prev_kw = {"monitor","inspect","preventive","prevent","continue",
                   "maintain","prune","mulch","rotation","certified","irrigation"}
        cleaned = [s for s in result.get("action_plan", [])
                   if any(kw in s.lower() for kw in prev_kw)]
        if not cleaned:
            cleaned = [
                "Continue weekly field monitoring for early disease detection",
                "Apply preventive neem-based spray before monsoon season",
                "Maintain proper plant spacing and drainage for air circulation",
                "Use balanced NPK fertilizer to maintain plant immunity",
            ]
        result["action_plan"] = cleaned
        result["marketplace"]["product_type"] = "Preventive"
        result["marketplace"]["note"] = "No immediate purchase required. Optional preventive measures only."

    # ── Rule 3: Confidence-aware tone ────────────────────────────
    if not is_healthy:
        if confidence >= 0.95:
            pass  # High confidence → decisive, no changes
        elif confidence >= 0.70:
            caveat = (
                f" [Confidence: {confidence*100:.1f}% — Monitor closely. "
                "Consider re-scanning with a clearer image to confirm.]"
            )
            result["first_aid"] = result["first_aid"].rstrip(".") + caveat
        else:
            # Low confidence — prepend uncertainty
            result["first_aid"] = (
                f"⚠️ LOW CONFIDENCE ({confidence*100:.1f}%): Model is uncertain. "
                "Take another photo with better lighting. Consult local agricultural officer "
                "before applying any chemicals. | " + result["first_aid"]
            )
            result["action_plan"] = [
                "Re-scan with a clearer, well-lit photo of the affected leaf",
                "Consult Krishi Vigyan Kendra expert (KVK Helpline: 1800-180-1551)"
            ] + result.get("action_plan", [])

    # ── Rule 6: Economic consistency ─────────────────────────────
    if is_healthy:
        result["yield_loss_pct"]        = 0
        result["economic_loss_rs"]       = 0.0
        result["economic_loss_per_acre"] = 0.0
    else:
        # Ensure loss > 0 for diseased plants
        if result.get("yield_loss_pct", 0) == 0:
            floor = {"LOW": 5, "MEDIUM": 20, "HIGH": 40}
            result["yield_loss_pct"] = floor.get(sev_level, 10)
        if result.get("economic_loss_rs", 0) == 0 and result["yield_loss_pct"] > 0:
            result["economic_loss_rs"]       = round(result["yield_loss_pct"] * 1000, 2)
            result["economic_loss_per_acre"] = result["economic_loss_rs"]

    # ── Rule 7: Sanity — tone down alarmism for LOW/low-confidence ──
    if sev_level in ("LOW", "NONE") or confidence < 0.70:
        for field in ("first_aid",):
            val = result.get(field, "")
            for word, replace in [("EMERGENCY", "ATTENTION NEEDED"),
                                   ("immediately", "promptly"),
                                   ("DESTROY ALL", "Remove affected")]:
                if word in val and sev_level not in ("HIGH",):
                    result[field] = val.replace(word, replace)

    return result

# ══════════════════════════════════════════════════════════════════════════
# WEATHER ADVICE  (dynamic, type-based)
# ══════════════════════════════════════════════════════════════════════════

def get_weather_advice(disease_key, lat, lon, api_key):
    k = disease_key.lower()
    is_healthy = "healthy" in k

    if is_healthy:
        return "☀️ Plant is healthy. Continue monitoring. Track weather for disease-favorable conditions."

    # Try live weather API
    if lat and lon and api_key:
        try:
            url = (f"https://api.openweathermap.org/data/2.5/forecast"
                   f"?lat={lat}&lon={lon}&appid={api_key}&cnt=3&units=metric")
            r = requests.get(url, timeout=5).json()
            fs = r.get("list", [])
            if fs:
                rain = any("rain" in f.get("weather",[{}])[0].get("description","").lower() for f in fs[:3])
                temp = sum(f["main"]["temp"]     for f in fs[:3]) / len(fs[:3])
                hum  = sum(f["main"]["humidity"] for f in fs[:3]) / len(fs[:3])
                parts = []
                parts.append("🌧️ Rain expected — delay spraying (wait for dry window)." if rain
                             else "☀️ Dry forecast — good window for spray applications.")
                if hum > 80:  parts.append(f"💧 High humidity ({hum:.0f}%) — fungal risk elevated.")
                if hum < 40:  parts.append(f"🌵 Low humidity ({hum:.0f}%) — spider mite risk elevated.")
                if temp > 35: parts.append(f"🌡️ High temp ({temp:.1f}°C) — spray before 8 AM.")
                if temp < 15: parts.append(f"❄️ Low temp ({temp:.1f}°C) — pesticide less effective below 18°C.")
                return " | ".join(parts)
        except:
            pass

    # Static advice based on disease type
    if any(x in k for x in _FUNGAL):
        return ("🌧️ Fungal diseases thrive in humid/wet conditions. "
                "Spray early morning (6–9 AM). Avoid spraying if rain expected within 4 hours.")
    if any(x in k for x in _VIRAL):
        return ("🦟 Viral disease — spread by insect vectors (whiteflies/aphids). "
                "Install yellow sticky traps. Spray insecticide in evening.")
    if any(x in k for x in _PEST):
        return ("🌡️ Spider mites thrive in hot, dry weather. "
                "Increase irrigation. Spray miticide in early morning.")
    if not lat:
        return "📍 Add GPS coordinates for personalized weather advice."
    return "🌤️ Apply treatments in calm, dry conditions. Early morning or late evening is best."

# ══════════════════════════════════════════════════════════════════════════
# DB INSERT  (Supabase — disease_scans table)
# ══════════════════════════════════════════════════════════════════════════

def insert_to_db(result):
    """
    Field mapping:
      result["timestamp"]        → scanned_at
      result["location"]["lat"]  → latitude
      result["location"]["lon"]  → longitude
      result["disease"]          → pathogen_name
      result["confidence_raw"]   → confidence_score
      is_positive                → True if "healthy" NOT in disease_key
    """
    if not SUPABASE_URL or not SUPABASE_KEY:
        print("⚠️  DB: Supabase credentials not set — skipping DB insert (dry-run mode).")
        print("     Set SUPABASE_URL and SUPABASE_KEY at the top of this cell to enable.")
        return None
    try:
        from supabase import create_client
        client = create_client(SUPABASE_URL, SUPABASE_KEY)
        loc    = result.get("location", {})
        record = {
            "scanned_at":      result.get("timestamp"),
            "latitude":        loc.get("lat"),
            "longitude":       loc.get("lon"),
            "pathogen_name":   result.get("disease") or result.get("disease_key"),
            "confidence_score": result.get("confidence_raw", 0.0),
            "is_positive":     result.get("is_positive", True),
            "severity_level":  result.get("severity_level", "UNKNOWN"),
            "disease_type":    result.get("disease_type", "unknown"),
            "yield_loss_pct":  result.get("yield_loss_pct"),
            "economic_loss_rs": result.get("economic_loss_rs"),
            "raw_payload":     result,
        }
        resp = client.table("disease_scans").insert(record).execute()
        inserted = resp.data[0] if resp.data else {}
        print(f"✅ DB: Scan saved! ID = {inserted.get('id', 'N/A')}")
        return inserted
    except ImportError:
        print("⚠️  DB: supabase package not installed. Run: pip install supabase")
    except Exception as e:
        print(f"❌ DB: Insert failed — {e}")
    return None

# ══════════════════════════════════════════════════════════════════════════
# MAIN INFERENCE + INTELLIGENCE PIPELINE
# ══════════════════════════════════════════════════════════════════════════

def run_pipeline(send_to_db=False):
    clear_output(wait=True)
    display(header_html)

    # ── Load protocols ─────────────────────────────────────────────
    with open(PROTOCOLS_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    protocols = data["protocols"]

    # ── Layer 1: Run model inference ───────────────────────────────
    with torch.no_grad():
        output = model(input_tensor)
        probs  = torch.softmax(output, dim=1)[0]

    k = min(TOP_K, len(CLASS_NAMES))
    topk_probs, topk_indices = torch.topk(probs, k)

    pred_idx    = topk_indices[0].item()
    confidence  = float(topk_probs[0].item())
    disease_key = CLASS_NAMES[pred_idx]

    # ── Detect disease context ─────────────────────────────────────
    context = detect_context(disease_key)
    context["confidence_raw"] = confidence

    # ── Get protocol ───────────────────────────────────────────────
    protocol = protocols.get(disease_key)
    if not protocol:
        print(f"⚠️ Unknown disease key: {disease_key}. Please check class names.")
        return

    # ── Economic impact ────────────────────────────────────────────
    base_loss   = protocol.get("economic_loss_per_acre_rs", 0)
    adjusted    = base_loss * (MARKET_PRICE_RS / 1500.0)
    total_loss  = adjusted * CROP_AREA_ACRES

    # ── Build raw result ───────────────────────────────────────────
    result = {
        "disease":           protocol["display_name"],
        "disease_key":       disease_key,
        "crop":              protocol["crop"],
        "confidence":        round(confidence * 100, 2),
        "confidence_raw":    round(confidence, 6),
        "severity":          "",
        "severity_level":    "",
        "first_aid":         protocol["first_aid"],
        "action_plan":       list(protocol["action_plan"]),
        "weather_advice":    get_weather_advice(disease_key, LATITUDE, LONGITUDE, OPENWEATHER_KEY),
        "yield_loss_pct":    protocol["yield_loss_pct"],
        "economic_loss_rs":  round(total_loss, 2),
        "economic_loss_per_acre": round(adjusted, 2),
        "marketplace": {
            "recommended_products": protocol["marketplace"]["recommended_products"],
            "product_type":         protocol["marketplace"]["product_type"],
            "note": "Contact your nearest Krishi Seva Kendra or AgriShop for availability.",
        },
        "location": {"lat": LATITUDE, "lon": LONGITUDE},
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "is_positive":  not context["is_healthy"],
        "disease_type": context["disease_type"],
        "top_k_predictions": [
            {"rank": i+1, "disease_key": CLASS_NAMES[topk_indices[i].item()],
             "confidence": round(float(topk_probs[i].item()) * 100, 2)}
            for i in range(k)
        ],
    }

    # ── Layer 2: Run Validation Engine (all 8 rules) ───────────────
    result = validate_result(result, context)

    # ── Print Report ───────────────────────────────────────────────
    print_report(result)

    # ── DB Insert ──────────────────────────────────────────────────
    if send_to_db:
        print("\n📤 Sending to Supabase DB...")
        insert_to_db(result)
    else:
        print("\n💡 DB insert is OFF (checkbox unchecked). Check the box to send data to database.")

    # ── Raw JSON ───────────────────────────────────────────────────
    print("\n" + "─"*65)
    print("📦 RAW JSON PAYLOAD (for backend/DB):")
    print("─"*65)
    print(json.dumps(result, indent=2, ensure_ascii=False))

def print_report(r):
    """Prints human-readable intelligence report."""
    sev_level = r.get("severity_level", "")
    sev_color = {"HIGH": "#ff4444", "MEDIUM": "#ffaa00", "LOW": "#44bb44",
                 "NONE": "#00cc88", "UNKNOWN": "#aaaaaa"}.get(sev_level, "#ffffff")

    print("═"*65)
    print("  🌿  CROP DISEASE INTELLIGENCE REPORT")
    print("═"*65)
    print(f"  🦠  Disease       : {r['disease']}")
    print(f"  🌾  Crop          : {r.get('crop','N/A')}")
    print(f"  📊  Confidence    : {r['confidence']}%")
    print(f"  🔬  Disease Type  : {r.get('disease_type','N/A').upper()}")
    print(f"  🚨  Severity      : {r['severity']}")
    print(f"  ✅  Is Positive   : {'YES — Disease found' if r.get('is_positive') else 'NO — Plant is Healthy'}")
    print()
    print("  💊  FIRST-AID REMEDY:")
    print(f"      {r['first_aid']}")
    print()
    print("  📋  GRANULAR ACTION PLAN:")
    for step in r.get("action_plan", []):
        print(f"      ▸ {step}")
    print()
    print("  🌦️  WEATHER-CONTEXTUAL ADVICE:")
    print(f"      {r['weather_advice']}")
    print()
    print("  📉  YIELD & ECONOMIC IMPACT:")
    if r.get("yield_loss_pct") is not None:
        print(f"      Expected Yield Loss  : {r['yield_loss_pct']}%")
        print(f"      Estimated Total Loss : ₹{r['economic_loss_rs']:,.0f}")
        print(f"      Per-Acre Loss        : ₹{r['economic_loss_per_acre']:,.0f}")
    else:
        print("      N/A")
    print()
    print("  🛒  MARKETPLACE ROUTING:")
    mp = r.get("marketplace", {})
    for p in mp.get("recommended_products", []):
        print(f"      🔹 {p}")
    print(f"      📌 {mp.get('note','')}")
    print()
    if r.get("top_k_predictions"):
        print("  🏆  TOP-K PREDICTIONS:")
        for alt in r["top_k_predictions"]:
            bar = "█" * int(alt["confidence"] / 5)
            print(f"      #{alt['rank']} [{bar:<20}] {alt['confidence']:5.1f}%  {alt['disease_key']}")
    print()
    loc = r.get("location", {})
    if loc.get("lat"):
        print(f"  📍  Location  : {loc['lat']}, {loc['lon']}")
    print(f"  🕐  Timestamp : {r.get('timestamp','N/A')}")
    print("═"*65)

# ══════════════════════════════════════════════════════════════════════════
# ☑️ CHECKBOX UI — Interactive widget
# ══════════════════════════════════════════════════════════════════════════

header_html = HTML("""
<div style="background: linear-gradient(135deg, #1a472a, #2d6a4f);
            padding: 16px 20px; border-radius: 12px; margin-bottom: 12px;
            font-family: 'Segoe UI', sans-serif; color: white;">
  <h2 style="margin:0; font-size:20px;">🌿 Crop Disease Intelligence Pipeline</h2>
  <p style="margin:4px 0 0 0; opacity:0.85; font-size:13px;">
    Layer 1 (MobileNetV2) + Layer 2 (Intelligence Engine + Validation)
  </p>
</div>
""")

# Checkbox: Send to DB
cb_db = widgets.Checkbox(
    value=False,
    description="📤 Send result to Supabase DB",
    style={"description_width": "initial"},
    layout=widgets.Layout(margin="4px 0"),
)

# Checkbox: Show raw JSON
cb_json = widgets.Checkbox(
    value=True,
    description="📦 Show raw JSON payload",
    style={"description_width": "initial"},
    layout=widgets.Layout(margin="4px 0"),
)

# Run button
btn_run = widgets.Button(
    description = "▶  Run Analysis",
    button_style= "success",
    icon        = "leaf",
    layout      = widgets.Layout(width="200px", height="40px", margin="10px 0"),
)
btn_run.style.button_color = "#2d6a4f"
btn_run.style.font_weight  = "bold"

output_area = widgets.Output()

def on_run(b):
    with output_area:
        run_pipeline(send_to_db=cb_db.value)

btn_run.on_click(on_run)

# ── Display UI ─────────────────────────────────────────────────────
display(header_html)
display(widgets.VBox([
    widgets.HTML("<b style='font-size:14px;'>⚙️ Options:</b>"),
    cb_db,
    widgets.HTML(
        "<span style='font-size:11px; color:#888;'>"
        "DB insert requires SUPABASE_URL and SUPABASE_KEY set at top of cell.</span>"
    ),
    btn_run,
]))
display(output_area)

# ── Also run immediately on cell execution (without clicking button) ──
with output_area:
    run_pipeline(send_to_db=False)


Output()